# Train the Rater — selectable outer/meta loss

This notebook trains only the Rater and saves a reusable `final_rater.pt`.
The final classifier remains a later experiment.

Representation and inner training:

- frozen Waterbirds-ERM ResNet-50 2048-D embeddings;
- two independent random-from-scratch `Linear(2048,2)` inner classifiers;
- every training example receives a Rater-derived weight;
- each inner classifier trains `INNER_STEPS` minibatches per Rater update (set in cell 6; 0 = one full training pass);
- 500 meta steps by default.

## Outer/meta-loss choices

Set `OUTER_LOSS` in cell 6 to one of:

- `standard_ce`: mean CE over all outer/meta examples.
- `worst_group_loss`: compute mean CE for each observed group in the outer/meta split, then take the largest group mean. This uses group annotations as a group-supervised meta-training ablation.
- `group_balanced_loss`: compute mean CE for each observed group, then average group means equally. This also uses group annotations.
- `classwise_top_loss`: average the top `OUTER_TOP_LOSS_PERCENT` CE losses within each class, then average the classes equally.
- `class_balanced`: average CE within each class, then average the class means equally.
- `worst_class_loss`: take the largest class mean CE.
- `logsumexp`: normalized log-sum-exp over all outer examples,
  `(logsumexp(beta * CE) - log(N)) / beta`. Larger `OUTER_LSE_BETA`
  emphasizes high-loss examples.
- `top_loss`: CVaR over all outer examples: the mean of the top
  `OUTER_TOP_LOSS_PERCENT` CE losses, classes ignored.
- `worst_class_top_loss`: like `classwise_top_loss`, but takes the worst class's
  tail mean instead of the average over classes.
- `classwise_logsumexp`: the normalized log-sum-exp within each class (same
  `OUTER_LSE_BETA`), then the classes averaged equally.
- `mean_plus_std`: mean CE + `OUTER_STD_COEF` × standard deviation of CE
  (variance-regularized objective).

Group annotations are read from the outer/meta dataset only for the two group-based objectives. The inner Rater weighting rule is unchanged. Groups absent from the outer split are excluded from the group aggregation.


In [ ]:
# 1. Paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import os, sys, subprocess, shutil, re, torch

# Shared project folder on Drive. Renamed to "FeatureRater" (Nov 2026); a Drive shortcut
# may still carry the old name, so both are accepted (the folder must contain archives/).
DRIVE_ROOT_NAMES = ("FeatureRater", "evidential_alignment_colab")
DRIVE_ROOT = next(
    (Path("/content/drive/MyDrive") / name for name in DRIVE_ROOT_NAMES
     if (Path("/content/drive/MyDrive") / name / "archives").exists()),
    Path("/content/drive/MyDrive") / DRIVE_ROOT_NAMES[-1],
)
print("Project folder on Drive:", DRIVE_ROOT)
REPO_DIR = Path("/content/evidential_alignment")
DATA_ROOT = Path("/content/datasets")
WB_DIR = DATA_ROOT / "waterbird_complete95_forest2water2"
WB_ARCHIVE = DRIVE_ROOT / "archives" / "waterbird_complete95_forest2water2.tar.gz"


# Drive layout (Nov 2026): waterbirds/<ERM_ID>/ holds the ERM checkpoint, its embeddings,
# raters/<run>/classifiers/, controls/, baselines/ and analysis/. Every path below falls
# back to the pre-November location, so this works before and after the reorganization.
ERM_ID = "erm-r50-e100-s0"
ERM_DIR = DRIVE_ROOT / "waterbirds" / ERM_ID
ERM_STABLE_NAME = "waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt"
ERM_RUN_NAME = "erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained"
CACHE_SUBDIR = "waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224"

def first_existing(*paths):
    """First path that exists; the first one if none does."""
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return Path(paths[0])

# Pins keep a moved checkpoint's original embedding-cache name (see RATER_ERM_FINGERPRINT
# in algorithms/rater.py); the md5 proves it is the same file. Unknown ERM_ID = no pin.
ERM_PINS = {"erm-r50-e100-s0": ("1055b1e8de29", "f28f7a0ddbc74dac7723eb09d2848001")}
ERM_FINGERPRINT, ERM_MD5 = ERM_PINS.get(ERM_ID, ("", ""))

ERM_MODEL = first_existing(
    ERM_DIR / ERM_STABLE_NAME,
    ERM_DIR / ERM_RUN_NAME / "best_val_acc_model.pt",
    DRIVE_ROOT / "erm_base_models" / ERM_STABLE_NAME,
    DRIVE_ROOT / "erm_base_models" / ERM_RUN_NAME / "best_val_acc_model.pt",
)
assert ERM_MODEL.exists(), ERM_MODEL

NEW_LAYOUT = ERM_DIR.exists()
EMBED_ROOT = ERM_DIR / "embeddings" if NEW_LAYOUT else DRIVE_ROOT / "embeddings_waterbirds_erm"
OUTPUT_ROOT = ERM_DIR / "raters" if NEW_LAYOUT else DRIVE_ROOT / "logsumexp_runs"
LOCAL_PRE = Path("/content/rater_precompute_local")
LOCAL_RUN = Path("/content/rater_meta_only_local")

for p in [WB_ARCHIVE.parent, EMBED_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("ERM:", ERM_MODEL)
print("Embedding cache:", EMBED_ROOT)
print("Rater outputs:", OUTPUT_ROOT)

Mounted at /content/drive
ERM: /content/drive/MyDrive/evidential_alignment_colab/erm_base_models/waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm
Rater outputs: /content/drive/MyDrive/evidential_alignment_colab/logsumexp_runs


In [ ]:
# 2. Fresh repo + dependencies
# Branch to clone. Set to "main" once this branch is merged into main.
REPO_BRANCH = "station/featurerater-cleanup"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--branch", REPO_BRANCH,
    "https://github.com/TheRealMmd/evidential_alignment.git",
    str(REPO_DIR),
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "transformers", "wilds", "wget", "matplotlib", "scipy",
], check=True)

REPO_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("✓ repo ready:", REPO_BRANCH, REPO_COMMIT)

✓ repo ready


In [ ]:
# 3. Waterbirds
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not (WB_ARCHIVE.exists() and WB_ARCHIVE.stat().st_size > 100_000_000):
    subprocess.run([
        "wget", "-c",
        "https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz",
        "-O", str(WB_ARCHIVE),
    ], check=True)

if not (WB_DIR / "metadata.csv").exists():
    subprocess.run([
        "tar", "-xzf", str(WB_ARCHIVE), "-C", str(DATA_ROOT)
    ], check=True)

assert (WB_DIR / "metadata.csv").exists()
print("✓ Waterbirds ready")

✓ Waterbirds ready


In [ ]:
# 4. Patch data path + PyTorch 2.6 loading
config_path = REPO_DIR / "config.py"
text = config_path.read_text(encoding="utf-8")
placeholder = '"waterbirds": "/path/to/datasets/waterbirds"'
if placeholder in text:
    text = text.replace(placeholder, f'"waterbirds": "{WB_DIR}"')
else:
    text, n = re.subn(
        r'("waterbirds"\s*:\s*)["\'][^"\']*["\']',
        lambda m: m.group(1) + '"' + str(WB_DIR) + '"',
        text,
        count=1,
    )
    assert n == 1
config_path.write_text(text, encoding="utf-8")

algorithm_path = REPO_DIR / "algorithms" / "algorithm.py"
text = algorithm_path.read_text(encoding="utf-8")
text = text.replace(
    'torch.load(check_point, map_location="cpu")',
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
text = text.replace(
    "torch.load(check_point, map_location='cpu')",
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
algorithm_path.write_text(text, encoding="utf-8")
print("✓ repo patched")

✓ repo patched


In [ ]:
# 5. Install the Rater implementation
#
# USE_REPO_RATER = True (default): run algorithms/rater.py exactly as committed
# in REPO_BRANCH, so every run is traceable to a commit. The old flow below
# reused a Drive copy (code/rater_classwise_top_loss.py) whenever it contained
# the outer-loss names, which silently skipped later diagnostics.
import hashlib

USE_REPO_RATER = True

if USE_REPO_RATER:
    rater_path = REPO_DIR / "algorithms" / "rater.py"
    RATER_SHA256 = hashlib.sha256(rater_path.read_bytes()).hexdigest()
    print("Using the repository's rater.py:", REPO_BRANCH, REPO_COMMIT)
    print("sha256:", RATER_SHA256)
else:
    # Legacy: upload algorithms/rater.py the first time, or whenever it changes.
    # The notebook stores the uploaded source on Drive for reuse.
    import ast
    from google.colab import files

    CODE_DIR = DRIVE_ROOT / "code"
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    SAVED_RATER_SOURCE = CODE_DIR / "rater_classwise_top_loss.py"

    REQUIRED_SOURCE_MARKERS = [
        "RATER_OUTER_LOSS",
        "_compute_outer_objective",
        "standard_ce",
        "worst_group_loss",
        "group_balanced_loss",
        "class_balanced",
        "worst_class_loss",
        "logsumexp",
    ]

    def is_current_rater_source(source):
        return all(marker in source for marker in REQUIRED_SOURCE_MARKERS)

    source_text = ""
    if SAVED_RATER_SOURCE.exists():
        candidate_text = SAVED_RATER_SOURCE.read_text(encoding="utf-8")
        if is_current_rater_source(candidate_text):
            source_text = candidate_text
            print("Using current saved Rater source:", SAVED_RATER_SOURCE)

    if not source_text:
        print("Upload algorithms/rater.py from this project.")
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.endswith(".py")]
        if len(candidates) != 1:
            raise RuntimeError("Upload exactly one Python file: algorithms/rater.py")
        source_text = uploaded[candidates[0]].decode("utf-8")
        if not is_current_rater_source(source_text):
            raise RuntimeError(
                "The uploaded rater.py does not include all outer-loss options."
            )
        SAVED_RATER_SOURCE.write_text(source_text, encoding="utf-8")
        print("Saved current Rater source to:", SAVED_RATER_SOURCE)

    ast.parse(source_text)

    rater_path = REPO_DIR / "algorithms" / "rater.py"
    rater_path.write_text(source_text, encoding="utf-8")

    init_path = REPO_DIR / "algorithms" / "__init__.py"
    init_text = init_path.read_text(encoding="utf-8")
    if "from . import rater" not in init_text:
        init_path.write_text(init_text.rstrip() + "\nfrom . import rater\n", encoding="utf-8")

    print("Rater code with selectable outer losses installed.")


Upload algorithms/rater.py from this project.


Saving rater.py to rater.py
Saved current Rater source to: /content/drive/MyDrive/evidential_alignment_colab/code/rater_classwise_top_loss.py
Rater code with selectable outer losses installed.


In [ ]:
# 6. Experiment parameters
DATASET = "waterbirds"
RATER_SEED = 0
SPLIT_VAL = 0.4     # D_meta = val_subset1 = this fraction of the validation set
SPLIT_TRAIN = 1.0   # < 1 creates train subsets (needed for a train-holdout D_meta)
TRAIN_SPLIT = "train"  # "train_subset1" when OUTER_SPLIT is a train holdout

META_STEPS = 500
BATCH_SIZE = 128
NUM_INNER_MODELS = 2
INNER_STEPS = 0                # minibatches per inner head before each rater update; 0 = one full pass
STEP_DIAGNOSTICS = False       # cheap visited-batch/outer metrics, no extra evaluation passes

# Choose: standard_ce, worst_group_loss, group_balanced_loss,
#         classwise_top_loss, class_balanced, worst_class_loss, logsumexp,
#         top_loss, worst_class_top_loss, classwise_logsumexp, mean_plus_std
OUTER_LOSS = "logsumexp"
OUTER_TOP_LOSS_PERCENT = 30.0  # used by classwise_top_loss, top_loss, worst_class_top_loss
OUTER_LSE_BETA = 1.0           # used by logsumexp, classwise_logsumexp
OUTER_STD_COEF = 1.0           # used only by mean_plus_std

assert OUTER_LOSS in {
    "standard_ce", "worst_group_loss", "group_balanced_loss",
    "classwise_top_loss", "class_balanced", "worst_class_loss", "logsumexp",
    "top_loss", "worst_class_top_loss", "classwise_logsumexp", "mean_plus_std",
}
assert 0.0 < OUTER_TOP_LOSS_PERCENT <= 100.0
assert OUTER_LSE_BETA > 0.0
assert OUTER_STD_COEF >= 0.0

RATER_WEIGHTING = "sigmoid"
INNER_LR = 2e-4
TEMPERATURE = 1.0
OUTER_LR = 3e-4
GRAD_CLIP = 5.0

# Diagnostics: evaluate often, but avoid hundreds of PNGs.
EVAL_FREQ = 10
PLOT_FREQ = 25

# Optional controls (defaults = previous behaviour). See algorithms/rater.py.
UNIFORM_REFERENCE = False      # uniform-weight reference heads next to the rater-weighted ones
REFRESH_STEPS = 0              # > 0: re-initialize each inner head every K meta steps (staggered)
SELECTION = "min_outer_loss"   # "fresh_heads" (not confounded by head age) or "last"
FRESH_HEAD_EPOCHS = 1
FRESH_HEAD_EVAL = False        # log the fresh-head objective even if not selecting by it
OUTER_SPLIT = ""               # e.g. "train_no_aug_subset2" with SPLIT_TRAIN < 1 (skewed D_meta)
OUTER_MAX_SAMPLES = 0          # cap the D_meta size (meta-set size sweep)

assert isinstance(INNER_STEPS, int) and INNER_STEPS >= 0
os.environ.update({
    "RATER_ERM_MODEL": str(ERM_MODEL),
    "RATER_ERM_FINGERPRINT": ERM_FINGERPRINT,
    "RATER_ERM_MD5": ERM_MD5,
    "RATER_EMBEDDING_CACHE_ROOT": str(EMBED_ROOT),
    "RATER_META_STEPS": str(META_STEPS),
    "RATER_INNER_STEPS": str(INNER_STEPS),
    "RATER_STEP_DIAGNOSTICS": "1" if STEP_DIAGNOSTICS else "0",
    "RATER_NUM_INNER_MODELS": str(NUM_INNER_MODELS),
    "RATER_WEIGHTING": RATER_WEIGHTING,
    "RATER_INNER_LR": str(INNER_LR),
    "RATER_OUTER_LR": str(OUTER_LR),
    "RATER_TEMPERATURE": str(TEMPERATURE),
    "RATER_GRAD_CLIP": str(GRAD_CLIP),
    "RATER_OUTER_LOSS": OUTER_LOSS,
    "RATER_OUTER_TOP_LOSS_PERCENT": str(OUTER_TOP_LOSS_PERCENT),
    "RATER_OUTER_LSE_BETA": str(OUTER_LSE_BETA),
    "RATER_OUTER_STD_COEF": str(OUTER_STD_COEF),
    "RATER_EVAL_FREQ": str(EVAL_FREQ),
    "RATER_PLOT_FREQ": str(PLOT_FREQ),
    "RATER_UNIFORM_REFERENCE": "1" if UNIFORM_REFERENCE else "0",
    "RATER_REFRESH_STEPS": str(REFRESH_STEPS),
    "RATER_SELECTION": SELECTION,
    "RATER_FRESH_HEAD_EPOCHS": str(FRESH_HEAD_EPOCHS),
    "RATER_FRESH_HEAD_EVAL": "1" if FRESH_HEAD_EVAL else "0",
    "RATER_OUTER_SPLIT": OUTER_SPLIT,
    "RATER_OUTER_MAX_SAMPLES": str(OUTER_MAX_SAMPLES),
})

print({
    "dataset": DATASET,
    "seed": RATER_SEED,
    "selection": SELECTION,
    "uniform_reference": UNIFORM_REFERENCE,
    "refresh_steps": REFRESH_STEPS,
    "outer_split": OUTER_SPLIT or f"val_subset1 (split_val={SPLIT_VAL})",
    "meta_steps": META_STEPS,
    "batch_size": BATCH_SIZE,
    "inner_models": NUM_INNER_MODELS,
    "inner_batches_per_update": INNER_STEPS or "full pass",
    "step_diagnostics": STEP_DIAGNOSTICS,
    "outer_loss": OUTER_LOSS,
    "outer_top_loss_percent_per_class": OUTER_TOP_LOSS_PERCENT,
    "outer_lse_beta": OUTER_LSE_BETA,
    "outer_uses_group_annotations": OUTER_LOSS in {"worst_group_loss", "group_balanced_loss"},
})


In [ ]:
# 7. Precompute/reuse persistent ERM embeddings
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "1"
shutil.rmtree(LOCAL_PRE, ignore_errors=True)
LOCAL_PRE.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", DATASET,
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_PRE),
    "--train_split", "train",
    "--split_val", str(SPLIT_VAL),
    "--split_train", str(SPLIT_TRAIN),
    "--mode", "train",
    "--epoch", str(META_STEPS),
    "--seed", str(RATER_SEED),
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Embedding preparation failed")

shutil.rmtree(LOCAL_PRE, ignore_errors=True)
print("✓ persistent ERM embeddings ready")

✓ persistent ERM embeddings ready


In [ ]:
# 8. Verify cache
cache_files = sorted(EMBED_ROOT.rglob("*.pt"))
for p in cache_files:
    print(p.relative_to(EMBED_ROOT), f"{p.stat().st_size / 1024**2:.2f} MB")

assert any(p.name == "train_no_aug.pt" for p in cache_files)
assert any("val_subset1" in p.name for p in cache_files)
assert any("val_subset2" in p.name for p in cache_files)
assert any(p.name == "test.pt" for p in cache_files)

sample = next(p for p in cache_files if p.name == "train_no_aug.pt")
payload = torch.load(sample, map_location="cpu", weights_only=False)
assert payload["cache_metadata"]["representation_source"] == "waterbirds_erm_checkpoint"
print("✓ cache verified:", tuple(payload["embeddings"].shape))

# ERM record: how the ERM checkpoint itself does on each cached split
# (written by the embedding step above; re-run cell 7 to create it for an older cache).
for record_csv in sorted(EMBED_ROOT.rglob("ERM_RECORD.csv")):
    print("\nERM record:", record_csv.relative_to(EMBED_ROOT))
    print(record_csv.read_text(encoding="utf-8"))

waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/test.pt 45.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/train_no_aug.pt 37.57 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val.pt 9.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p4_seed_0.pt 3.76 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_0.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_1.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_11.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_42.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_63.pt 5.64 MB
waterbirds/resnet50_water

In [ ]:
# 9. Run Rater meta-learning with the selected outer loss
rater_source = (REPO_DIR / "algorithms/rater.py").read_text()
assert all(marker in rater_source for marker in (
    "RATER_INNER_STEPS", "inner_batches_per_update", "step_diagnostics",
)), "Rerun cell 2 (Fresh repo + dependencies): the loaded rater.py is older than this notebook"
print("Inner batches per update:", INNER_STEPS or "full pass", "| fresh-head selection epochs:", FRESH_HEAD_EPOCHS)
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "0"
shutil.rmtree(LOCAL_RUN, ignore_errors=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)

# Rater hyperparameters live in environment variables, so the command is short.
cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", DATASET,
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_RUN),
    "--train_split", TRAIN_SPLIT,
    "--test_split", "test",
    "--split_val", str(SPLIT_VAL),
    "--split_train", str(SPLIT_TRAIN),
    "--batch_size", str(BATCH_SIZE),
    "--mode", "train",
    "--epoch", str(META_STEPS),
    "--seed", str(RATER_SEED),
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Rater meta-training failed")

print("✓ Rater meta-training complete")

In [ ]:
# 10. Copy only model + plots + log + summary to Drive
rater_files = sorted(LOCAL_RUN.rglob("final_rater.pt"))
assert len(rater_files) == 1, f"found {len(rater_files)} final_rater.pt files"

local_exp = rater_files[0].parent
for name in ["final_rater.pt", "log.txt", "summary.txt", "plots"]:
    assert (local_exp / name).exists(), name

import json
saved_config = json.loads((local_exp / "plots/data/run_config.json").read_text())
actual = saved_config["settings"]
assert actual["inner_batches_per_update"] == (INNER_STEPS or actual["full_epoch_batches"])
assert actual["step_diagnostics"] == STEP_DIAGNOSTICS
if STEP_DIAGNOSTICS:
    assert (local_exp / "plots/data/inner_step_metrics.csv").exists()
print("Verified actual inner batch budget:", actual["inner_batches_per_update"])

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
if OUTER_LOSS == "classwise_top_loss":
    objective_label = f"top{OUTER_TOP_LOSS_PERCENT:g}pct_per_class"
elif OUTER_LOSS == "logsumexp":
    beta_label = str(OUTER_LSE_BETA).replace(".", "p")
    objective_label = f"logsumexp_beta_{beta_label}"
elif OUTER_LOSS == "top_loss":
    objective_label = f"top{OUTER_TOP_LOSS_PERCENT:g}pct_all"
elif OUTER_LOSS == "worst_class_top_loss":
    objective_label = f"top{OUTER_TOP_LOSS_PERCENT:g}pct_worst_class"
elif OUTER_LOSS == "classwise_logsumexp":
    beta_label = str(OUTER_LSE_BETA).replace(".", "p")
    objective_label = f"classwise_logsumexp_beta_{beta_label}"
elif OUTER_LOSS == "mean_plus_std":
    objective_label = "mean_plus_" + f"{OUTER_STD_COEF:g}".replace(".", "p") + "std"
else:
    objective_label = OUTER_LOSS

budget_label = f"_inner{INNER_STEPS}" if INNER_STEPS else ""
drive_run = OUTPUT_ROOT / f"{objective_label}_all_sample{budget_label}_{stamp}"
drive_run.mkdir(parents=True, exist_ok=False)

for name in ["final_rater.pt", "log.txt", "summary.txt"]:
    shutil.copy2(local_exp / name, drive_run / name)
shutil.copytree(local_exp / "plots", drive_run / "plots")

print("Saved:", drive_run)
print("Rater:", drive_run / "final_rater.pt")
print("Plots:", drive_run / "plots")

In [ ]:
# 11. Summary + plot inventory
print((drive_run / "summary.txt").read_text(encoding="utf-8"))

plots = sorted((drive_run / "plots").rglob("*.png"))
print(f"Saved {len(plots)} PNG plots")
for p in plots:
    print(p.relative_to(drive_run))
print("Outer loss:", OUTER_LOSS)
print("Top-loss percentage (if selected):", OUTER_TOP_LOSS_PERCENT)
print("Log-sum-exp beta (if selected):", OUTER_LSE_BETA)
print("Outer objective uses group annotations:", OUTER_LOSS in {"worst_group_loss", "group_balanced_loss"})
